In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [ ]:
subs_df['original_comid'] = subs_df.index

# A small % of gauges are dropped if they don't match well.
common_idx = subs_df.index.intersection(gauges.index)
subs_df.loc[common_idx, 'original_comid'] = gauges.loc[common_idx, 'COMID']

subs_df

In [ ]:
datasets = Path("/nas/cee-ice/data")
glow_dir = datasets / "GLOW-S" / "daily_reach_aggregated"

glow_dfs = []
for region_file in glow_dir.glob('*_daily_median.parquet'):
    glow_dfs.append(pd.read_parquet(region_file))
                    
glow = pd.concat(glow_dfs)

In [ ]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched")
store = ZarrBasinStore(store_path)


def get_glow_s(comid):
    comid = int(comid)
    try:
        glow_ix = glow.xs(comid, level='COMID')
    except KeyError:
        return None
    
    glow_ix.index = pd.to_datetime(glow_ix.index).tz_localize('UTC')
    return glow_ix.rename(columns={'width':'glow_width'})

    

BATCH_SIZE = 50 
# Iterate over basins (distinct Zarr stores)
for basin_id, basin_group in tqdm(subs_df.groupby('outlet_id')):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)
    
    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        
        batch_df_list = []
        for subbasin in batch_subs:
            comid = basin_group.loc[subbasin]['original_comid']
            site_df = get_glow_s(comid)
                
            if site_df is not None and not site_df.empty:
                site_df['subbasin'] = subbasin
                batch_df_list.append(site_df)

        # If the whole batch is empty, we technically don't need to write anything 
        # since the zarr was initialized with NaNs/empty).
        if batch_df_list:
            batch_df = pd.concat(batch_df_list)
            store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)